#  — Architecture Integration

E-commerce AI Data Platform using the same Day 5 flow:

Kafka simulation → Bronze/Silver/Gold Lakehouse → Data Contracts → DAMA Quality Gate → RAG serving → Orchestration DAG with lineage.

In [ ]:
!pip install -q pandas pyarrow chromadb sentence-transformers loguru "pydantic>=2.0"

### Optional Groq API Key Setup

This notebook can run without Groq. If you have a Groq API key, add it in Colab Secrets as `GROQ_API_KEY`. If not, the RAG section will print a simple grounded answer from the retrieved context.

In [ ]:
import os
import json
import uuid
import random
from datetime import datetime, timedelta, UTC
from collections import defaultdict

import pandas as pd
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
from loguru import logger
from pydantic import BaseModel, Field, ValidationError, field_validator, ConfigDict

logger.add("capstone_run.log", rotation="10 MB")

BRONZE_PATH = "./lakehouse/bronze"
SILVER_PATH = "./lakehouse/silver"
GOLD_PATH   = "./lakehouse/gold"
QUARANTINE_PATH = "./lakehouse/quarantine"

random.seed(42)


In [ ]:
# Optional LLM client. The project still runs without it.
try:
    from groq import Groq
    try:
        from google.colab import userdata
        GROQ_API_KEY = userdata.get("GROQ_API_KEY")
    except Exception:
        GROQ_API_KEY = os.getenv("GROQ_API_KEY")

    groq_client = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None
    if groq_client:
        logger.info("Groq client initialized successfully.")
    else:
        logger.info("Groq key not found. Using fallback grounded answer.")
except Exception as e:
    groq_client = None
    logger.info(f"Groq is optional and was not initialized: {e}")


In [ ]:
# =====================================================================
# MODULE 1 — INGESTION (Kafka simulation)
# =====================================================================

class MockKafka:
    """
    Simulates Kafka's topic/partition/offset model.

    In real Kafka:
      - Topics are split into partitions.
      - Producers write events to a topic.
      - Consumers read events using offsets.
    """

    def __init__(self):
        self.topics  = defaultdict(list)
        self.offsets = defaultdict(int)

    def produce(self, topic: str, event: dict):
        self.topics[topic].append({
            "offset":     self.offsets[topic],
            "event_time": datetime.now(UTC).isoformat(),
            "payload":    event,
        })
        self.offsets[topic] += 1

    def consume_all(self, topic: str) -> list[dict]:
        return self.topics.pop(topic, [])


def module1_ingest(kafka: MockKafka) -> int:
    """
    Produces synthetic e-commerce product events to a Kafka-like topic.
    This replaces a real Kafka producer for the Colab project.
    """
    logger.info("[MODULE 1] Ingestion — producing events to Kafka...")

    products = [
        {"product_id": "P-100", "product_name": "Wireless Mouse", "category": "Accessories", "price": 79.0},
        {"product_id": "P-101", "product_name": "Mechanical Keyboard", "category": "Accessories", "price": 250.0},
        {"product_id": "P-102", "product_name": "Laptop Stand", "category": "Office", "price": 120.0},
        {"product_id": "P-103", "product_name": "USB-C Hub", "category": "Accessories", "price": 145.0},
        {"product_id": "P-104", "product_name": "Noise Cancelling Headset", "category": "Audio", "price": 399.0},
    ]

    review_templates = [
        "good quality and easy to use",
        "fast delivery and useful for work",
        "the product is okay but packaging can improve",
        "excellent value and works as expected",
        "battery life is good and setup was simple",
    ]

    for i in range(30):
        product = random.choice(products)
        event_type = random.choice(["product_view", "product_review", "inventory_update", "support_ticket"])

        event = {
            "event_id": f"EVT-{1000 + i}",
            "event_type": event_type,
            "product_id": product["product_id"],
            "product_name": product["product_name"],
            "category": product["category"],
            "price": product["price"],
            "rating": random.randint(3, 5) if event_type == "product_review" else None,
            "review_text": random.choice(review_templates) if event_type == "product_review" else None,
            "stock_quantity": random.randint(0, 80) if event_type == "inventory_update" else None,
            "ticket_text": "customer asked about availability and warranty" if event_type == "support_ticket" else None,
            "landed_at": datetime.now(UTC).isoformat(),
        }
        kafka.produce("ecommerce_raw", event)

    # Add two bad records to show Bronze is raw and Silver applies data contracts.
    kafka.produce("ecommerce_raw", {
        "event_id": "EVT-BAD-1",
        "event_type": "product_review",
        "product_id": "P-999",
        "product_name": "Unknown Product",
        "category": "Accessories",
        "price": -50.0,
        "rating": 5,
        "review_text": "invalid price example",
        "stock_quantity": None,
        "ticket_text": None,
        "landed_at": datetime.now(UTC).isoformat(),
    })

    kafka.produce("ecommerce_raw", {
        "event_id": "EVT-BAD-2",
        "event_type": "product_review",
        "product_id": "P-101",
        "product_name": "Mechanical Keyboard",
        "category": "Accessories",
        "price": 250.0,
        "rating": 9,
        "review_text": "invalid rating example",
        "stock_quantity": None,
        "ticket_text": None,
        "landed_at": datetime.now(UTC).isoformat(),
    })

    count = kafka.offsets["ecommerce_raw"]
    logger.success(f"[MODULE 1] {count} events published to ecommerce_raw")
    return count


In [ ]:
# =====================================================================
# DATA CONTRACT — Bronze to Silver validation
# =====================================================================

# TODO: Define the EcommerceEventContract using Pydantic.
# It should validate:
# - event_id
# - event_type
# - product_id
# - product_name
# - category
# - price
# - rating
# - landed_at

class EcommerceEventContract(BaseModel):
    """
    Data contract between the producer and the Silver consumer.

    This contract is applied after Bronze. Bronze keeps raw data, while
    Silver only accepts records that follow the expected schema and rules.
    """

    model_config = ConfigDict(extra="allow")

    event_id: str
    event_type: str
    product_id: str
    product_name: str
    category: str
    price: float = Field(gt=0)
    rating: int | None = None
    review_text: str | None = None
    stock_quantity: int | None = None
    ticket_text: str | None = None
    landed_at: str

    @field_validator("event_type")
    @classmethod
    def validate_event_type(cls, v):
        allowed = [
            "product_view",
            "product_review",
            "inventory_update",
            "support_ticket",
        ]
        if v not in allowed:
            raise ValueError("Invalid event_type")
        return v

    @field_validator("rating")
    @classmethod
    def validate_rating(cls, v):
        if v is not None and not (1 <= v <= 5):
            raise ValueError("rating must be between 1 and 5")
        return v

    @field_validator("stock_quantity")
    @classmethod
    def validate_stock(cls, v):
        if v is not None and v < 0:
            raise ValueError("stock_quantity must be >= 0")
        return v











In [ ]:
# =====================================================================
# MODULE 2 — STORAGE (Bronze → Silver → Gold)
# =====================================================================

def module2_storage(kafka: MockKafka) -> dict[str, str]:
    """
    Implements the Medallion Architecture:

      Bronze — Raw append: stores the events exactly as received.
      Silver — Validated and cleaned using Data Contracts.
      Gold   — Business-ready product knowledge for analytics and RAG.
    """
    logger.info("[MODULE 2] Storage — Bronze → Data Contract → Silver → Gold pipeline...")

    events = kafka.consume_all("ecommerce_raw")
    payloads = [e["payload"] for e in events]

    for path in [BRONZE_PATH, SILVER_PATH, GOLD_PATH, QUARANTINE_PATH]:
        os.makedirs(path, exist_ok=True)

    # Bronze: raw data, no cleaning
    bronze_df = pd.DataFrame(payloads)
    bronze_file = f"{BRONZE_PATH}/ecommerce_events_{int(datetime.now(UTC).timestamp())}.parquet"
    bronze_df.to_parquet(bronze_file, index=False)
    logger.info(f"[MODULE 2] Bronze: {len(bronze_df)} rows → {bronze_file}")

    # Silver: data contract validation
    valid_records = []
    invalid_records = []

    for record in payloads:
        try:
            valid_event = EcommerceEventContract(**record)
            valid_records.append(valid_event.model_dump())
        except ValidationError as e:
            bad_record = record.copy()
            bad_record["contract_error"] = str(e.errors())
            invalid_records.append(bad_record)

    silver_df = pd.DataFrame(valid_records)
    silver_file = f"{SILVER_PATH}/ecommerce_events_clean.parquet"
    silver_df.to_parquet(silver_file, index=False)

    quarantine_file = f"{QUARANTINE_PATH}/contract_rejected_records.parquet"
    pd.DataFrame(invalid_records).to_parquet(quarantine_file, index=False)

    logger.info(f"[MODULE 2] Silver: {len(silver_df)} rows passed the data contract")
    logger.info(f"[MODULE 2] Quarantine: {len(invalid_records)} rows rejected by the data contract")

    # Gold: prepare product-level knowledge for RAG
    reviews_df = silver_df[silver_df["event_type"] == "product_review"].copy()
    inventory_df = silver_df[silver_df["event_type"] == "inventory_update"].copy()
    tickets_df = silver_df[silver_df["event_type"] == "support_ticket"].copy()

    product_base = (
        silver_df[["product_id", "product_name", "category", "price"]]
        .drop_duplicates("product_id")
        .copy()
    )

    review_agg = (
        reviews_df
        .groupby("product_id")
        .agg(
            avg_rating=("rating", "mean"),
            review_count=("event_id", "count"),
            review_summary=("review_text", lambda x: "; ".join([str(v) for v in x.dropna().head(3)])),
        )
        .reset_index()
    ) if not reviews_df.empty else pd.DataFrame(columns=["product_id", "avg_rating", "review_count", "review_summary"])

    inventory_agg = (
        inventory_df
        .sort_values("landed_at")
        .groupby("product_id")
        .tail(1)[["product_id", "stock_quantity"]]
    ) if not inventory_df.empty else pd.DataFrame(columns=["product_id", "stock_quantity"])

    ticket_agg = (
        tickets_df
        .groupby("product_id")
        .agg(ticket_count=("event_id", "count"))
        .reset_index()
    ) if not tickets_df.empty else pd.DataFrame(columns=["product_id", "ticket_count"])

    gold_df = (
        product_base
        .merge(review_agg, on="product_id", how="left")
        .merge(inventory_agg, on="product_id", how="left")
        .merge(ticket_agg, on="product_id", how="left")
    )

    gold_df["avg_rating"] = gold_df["avg_rating"].fillna(0).round(2)
    gold_df["review_count"] = gold_df["review_count"].fillna(0).astype(int)
    gold_df["stock_quantity"] = gold_df["stock_quantity"].fillna(0).astype(int)
    gold_df["ticket_count"] = gold_df["ticket_count"].fillna(0).astype(int)
    gold_df["review_summary"] = gold_df["review_summary"].fillna("No reviews available yet")

    gold_df["rag_document"] = gold_df.apply(
        lambda row: (
            f"Product {row.product_name} ({row.product_id}) is in category {row.category}. "
            f"The price is {row.price} SAR. Current stock quantity is {row.stock_quantity}. "
            f"Average rating is {row.avg_rating} based on {row.review_count} reviews. "
            f"Support ticket count is {row.ticket_count}. Review notes: {row.review_summary}."
        ),
        axis=1,
    )

    gold_file = f"{GOLD_PATH}/product_knowledge.parquet"
    gold_df.to_parquet(gold_file, index=False)
    logger.success("[MODULE 2] Gold layer ready")

    return {
        "bronze": bronze_file,
        "silver": silver_file,
        "gold": gold_file,
        "quarantine": quarantine_file,
    }


In [ ]:
# =====================================================================
# MODULE 3 — QUALITY GATE (DAMA + Great Expectations-style)
# =====================================================================

def module3_quality(silver_path: str) -> bool:
    """
    Runs DAMA quality checks on the Silver layer before serving.

    DAMA dimensions used here:
      Completeness, Accuracy, Consistency, Timeliness, Uniqueness, Validity.
    """

    logger.info("[MODULE 3] Quality Gate — validating Silver layer with DAMA checks...")

    df = pd.read_parquet(silver_path)
    passed = True

    now = datetime.now(UTC)
    landed_at = pd.to_datetime(df["landed_at"], utc=True)

    # checks

    checks = {
        "Completeness": df[
            [
                "event_id",
                "event_type",
                "product_id",
                "product_name",
                "category",
                "price",
                "landed_at",
            ]
        ].notna().all().all(),

        "Accuracy": (df["price"] > 0).all(),

        "Consistency": df["event_type"].isin([
            "product_view",
            "product_review",
            "inventory_update",
            "support_ticket",
        ]).all(),

        "Timeliness": (
            (now - landed_at).dt.total_seconds() < 86400
        ).all(),

        "Uniqueness": df["event_id"].is_unique,

        "Validity": df["rating"].dropna().between(1, 5).all(),
    }

    report = []

    for dimension, result in checks.items():
        status = "PASS" if result else "FAIL"

        report.append({
            "dimension": dimension,
            "status": status,
            "checked_at": datetime.now(UTC).isoformat(),
        })

        if result:
            logger.success(f"[QUALITY] {dimension}: PASS")
        else:
            logger.error(f"[QUALITY] {dimension}: FAIL")
            passed = False

    os.makedirs("quality_reports", exist_ok=True)

    report_path = "quality_reports/dama_quality_report.json"

    with open(report_path, "w") as f:
        json.dump(report, f, indent=2)

    logger.info(f"[MODULE 3] Quality report saved to {report_path}")

    return passed



In [ ]:
# =====================================================================
# MODULE 4 — SERVING (RAG pipeline over Gold data)
# =====================================================================

def module4_rag_serve(gold_path: str):
    """
    Indexes Gold layer records into ChromaDB and answers product questions
    using Retrieval-Augmented Generation.
    """
    logger.info("[MODULE 4] Serving — building RAG index over Gold data...")
    df = pd.read_parquet(gold_path)

    documents = df["rag_document"].tolist()
    ids = df["product_id"].tolist()

    ef = SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
    client = chromadb.Client()

    try:
        client.delete_collection("ecommerce_gold")
    except Exception:
        pass

    collection = client.create_collection("ecommerce_gold", embedding_function=ef)
    collection.add(ids=ids, documents=documents)

    queries = [
        "Which product has the best customer rating?",
        "Which product has low stock?",
        "What do customers say about the keyboard?",
    ]

    for query in queries:
        results = collection.query(query_texts=[query], n_results=2)
        top_docs = results["documents"][0]
        context = "\n".join(top_docs)

        print(f"\n  🔍 RAG Query: {query}")
        print(f"  📄 Retrieved Context:\n{context}")

        if groq_client:
            try:
                chat_completion = groq_client.chat.completions.create(
                    messages=[
                        {
                            "role": "system",
                            "content": "Answer concisely using only the provided product context.",
                        },
                        {
                            "role": "user",
                            "content": f"Context:\n{context}\n\nQuestion: {query}",
                        },
                    ],
                    model="llama-3.1-8b-instant",
                    temperature=0,
                    max_tokens=160,
                )
                llm_answer = chat_completion.choices[0].message.content
                print(f"  🤖 LLM Answer: {llm_answer}")
            except Exception as e:
                logger.error(f"Error calling Groq API: {e}")
                print("  🤖 Fallback Answer:")
                print("  " + context.replace("\n", "\n  "))
        else:
            print("  🤖 Fallback Answer:")
            print("  " + context.replace("\n", "\n  "))

    logger.success("[MODULE 4] RAG serving complete")


In [ ]:
# =====================================================================
# MODULE 5 — ORCHESTRATION (Airflow-style DAG + OpenLineage-style events)
# =====================================================================

def module5_orchestrate():
    """
    Executes all modules in dependency order.

    In production: each function becomes an Airflow PythonOperator.
    Lineage events are written as a simple OpenLineage-style audit trail.
    """
    logger.info("[MODULE 5] DAG orchestrator starting...")
    run_id = str(uuid.uuid4())[:8]
    lineage = []

    def emit(task: str, event: str, details: dict = None):
        lineage.append({
            "runId": run_id,
            "task": task,
            "event": event,
            "ts": datetime.now(UTC).isoformat(),
            **(details or {}),
        })

    kafka = MockKafka()

    emit("ingest", "START")
    count = module1_ingest(kafka)
    emit("ingest", "COMPLETE", {"events_produced": count, "output_topic": "ecommerce_raw"})

    emit("storage", "START")
    paths = module2_storage(kafka)
    emit("storage", "COMPLETE", paths)

    emit("quality_gate", "START")
    quality_ok = module3_quality(paths["silver"])

    if quality_ok:
        emit("quality_gate", "COMPLETE")
        emit("rag_serve", "START")
        module4_rag_serve(paths["gold"])
        emit("rag_serve", "COMPLETE")
    else:
        emit("quality_gate", "FAIL", {"action": "quarantine — serve step skipped"})
        logger.error("[MODULE 5] Quality gate failed — downstream serve step skipped.")

    os.makedirs("lineage_events", exist_ok=True)
    lineage_path = f"lineage_events/run_{run_id}.json"
    with open(lineage_path, "w") as f:
        json.dump(lineage, f, indent=2)
    logger.info(f"[MODULE 5] Lineage → {lineage_path}")

    return quality_ok


In [ ]:
# =====================================================================
# ENTRY POINT
# =====================================================================

def main():
    print("=" * 70)
    print("  Day 5 Capstone — E-commerce AI Data Platform")
    print("=" * 70)

    success = module5_orchestrate()

    print("\n" + "=" * 70)
    if success:
        print("✅ All modules ran successfully end-to-end.")
        print("")
        print("   Output directories:")
        print("   • ./lakehouse/bronze/      — raw ingested events")
        print("   • ./lakehouse/silver/      — records accepted by data contract")
        print("   • ./lakehouse/quarantine/  — rejected records")
        print("   • ./lakehouse/gold/        — product knowledge for RAG")
        print("   • ./quality_reports/       — DAMA quality report")
        print("   • ./lineage_events/        — pipeline audit trail")
    else:
        print("⚠️ Pipeline completed but quality gate failed.")
        print("   Check the Silver data and the DAMA quality report.")


## ▶ Run

Run the next cell to execute the full Day 5-style capstone pipeline.

In [ ]:
main()